# Droplet：去除 doublet、整合 DNA/RNA QC 与 AnnData

填写下面两个 PJ 文件夹，确认 doublet 阈值倍率后从头运行。DNA 项目须已 complete，且已生成当前交付代的 `DNAme_QC_Information.csv`；RNA 项目须已有 finalized `RNA.h5ad`。

按文库分别用 RNA 原始 expected counts 运行 Scrublet，保留 doublet 分数与布尔列 `predicted_doublet`。整合表包含双方 QC 可用的共同细胞；singlet AnnData 只保留其中预测为 singlet 的细胞，`X` 和 `layers['counts']` 保留未取整原始计数，整合 QC 写入 `obs`。没有额外细胞 QC 阈值筛选。

完整对应表保留 DNA-only/RNA-only 细胞。没有 RNA 的细胞标记 `not_assessed_no_rna`，不视为 singlet。随后独立分析并标记全部 DNA 的 doublet；SingleCpG 保留全部 DNA 并加入 DNA/RNA/联合判定，不运行聚类或注释。

In [ ]:
RNA_PROJECT_ROOT = ""  # RNA PJ
DNA_PROJECT_ROOT = ""  # DNA PJ
DOUBLET_THRESHOLD_SCALE = 0.5
DNA_DOUBLET_THRESHOLD_SCALE = 0.5


## 运行整合

从当前 Notebook 所在 Lemmus 或受管 Python 内核自动定位 Processor，无需填写 Pipeline 路径。DNA manifest 的可选 `rna_sample` 填 RNA_DARLIN **原始 RNA basename**，与 Cabernet 一致；读取当前 complete DNA 的 sealed manifest，一对一解析 RNA raw → canonical sample_id。留空保持未关联，不按单文库、同名、排序或 barcode 重叠猜配。

Scrublet 使用默认先验 `expected_doublet_rate=0.05`、seed=0，按文库自动选评分阈值，再乘 `DOUBLET_THRESHOLD_SCALE`，仅剔除分数严格大于实际阈值的细胞。倍率必须在 `(0, 1]`：`1.0` 使用自动阈值，默认 `0.5` 更严格，越小剔除越多。5% 是模型先验，不是强制删除比例。降低阈值可能误删 singlet，不能保证清除全部 doublet；分数图同时显示自动与实际阈值。低信息矩阵或自动阈值失败会停止并给出原因。

In [ ]:
from pathlib import Path
import sys
import json

_candidates = []
for _start in (Path.cwd().resolve(), Path(sys.prefix).resolve()):
    for _base in (_start, *_start.parents):
        if (_base / "core/lemmus/downstream/qc.py").is_file():
            _candidates.append(_base)
            break
_candidates = list(dict.fromkeys(_candidates))
if len(_candidates) != 1:
    raise RuntimeError("请从 Lemmus/downstream 打开此 Notebook，并选择同一 Lemmus 的受管 atlas 内核。")
PIPELINE_ROOT = _candidates[0]
sys.path.insert(0, str(PIPELINE_ROOT / "core"))
from lemmus.downstream import doublets as processor
if Path(processor.__file__).resolve() != PIPELINE_ROOT / "core/lemmus/downstream/doublets.py":
    raise RuntimeError("内核已导入另一份 Processor，请重启内核。")

RESULT_DIR = processor.run_droplet_qc_integration(
    RNA_PROJECT_ROOT, DNA_PROJECT_ROOT, doublet_threshold_scale=DOUBLET_THRESHOLD_SCALE)
summary = json.loads((RESULT_DIR / "integration_summary.json").read_text())
print(f"RNA called cells: {summary['rna_cells']:,}")
print(f"共同 QC cells: {summary['integrated_cells']:,}")
print(f"共同细胞中预测 doublet: {summary['integrated_doublets']:,}")
print(f"最终 singlets: {summary['singlet_cells']:,}")
for library, info in summary["libraries"].items():
    print(f"{library}: 自动阈值 {info['auto_threshold']:.4f} → 实际阈值 {info['threshold']:.4f}; "
          f"全部 RNA doublet {info['n_doublets_auto']:,} → {info['n_doublets']:,}/{info['n_cells']:,}")
print(RESULT_DIR)

## 读取整合表与 singlet AnnData

`Integrated_QC_Information.csv` 是剔除前的共同 QC 表，包含 `doublet_score` 和 `predicted_doublet`；`Integrated_QC_singlets.csv` 与 `RNA_DNA_singlets.h5ad.obs` 一一对应。DNA/RNA 原有的 20/13 列发布表保持不变。输出保存在 RNA PJ 的 `06_downstream/<RNA身份>/droplet_integration/<本次整合身份>/`。

In [ ]:
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

qc = pd.read_csv(RESULT_DIR / "Integrated_QC_Information.csv",
                 dtype={"Sample_ID": str, "CloneID": str, "PlateID": str}, keep_default_na=False)
links = pd.read_csv(RESULT_DIR / "DNA_RNA_cell_links.csv", dtype=str, keep_default_na=False)
scores = pd.read_csv(RESULT_DIR / "RNA_doublet_scores.csv", dtype={"Sample_ID": str, "CloneID": str}, keep_default_na=False)
adata = ad.read_h5ad(RESULT_DIR / "RNA_DNA_singlets.h5ad")
assert adata.obs_names.tolist() == qc.loc[~qc.predicted_doublet, "Sample_ID"].tolist()
display(qc.head())
display(pd.DataFrame(summary["library_links"]))
print(adata)
FIGURES_DIR = RESULT_DIR / "plots"
FIGURES_DIR.mkdir(exist_ok=True)
sns.set_theme(style="ticks", context="notebook")
def save_plot(fig, name):
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / f"{name}.png", dpi=180, bbox_inches="tight")
    fig.savefig(FIGURES_DIR / f"{name}.pdf", bbox_inches="tight")
    plt.show()

## 细胞对应与 doublet 判定

左图按完整双模态细胞并集计数；中图按全部 RNA called cells 计数；右图展示各文库的 RNA doublet 评分、自动阈值（灰色点线）与实际剔除阈值（红色虚线）。不同图的分母不同，图标题中明确标出。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
links.membership.value_counts().plot.bar(ax=axes[0], color="#4C78A8", rot=25)
axes[0].set(title="DNA/RNA union membership", ylabel="Cells", xlabel="")
pd.crosstab(scores.CloneID, scores.predicted_doublet).rename(columns={False:"Singlet", True:"Doublet"}).plot.bar(
    stacked=True, ax=axes[1], color=["#59A14F", "#E15759"], rot=25)
axes[1].set(title="All RNA called cells", ylabel="Cells", xlabel="Library")
for library, info in summary["libraries"].items():
    values = scores.loc[scores.CloneID.eq(library), "doublet_score"]
    axes[2].hist(values, bins=40, histtype="step", label=library)
    axes[2].axvline(info["auto_threshold"], color="gray", linestyle=":", alpha=0.7, label=f"{library} auto")
    axes[2].axvline(info["threshold"], color="#E15759", linestyle="--", alpha=0.7, label=f"{library} applied")
axes[2].set(title="RNA Scrublet scores / thresholds", xlabel="Doublet score", ylabel="Cells")
axes[2].legend()
sns.despine(fig=fig)
save_plot(fig, "01_membership_doublets")

## 联合 QC 与 singlet QC 分布

散点图展示剔除前的共同细胞，红色标记预测 doublet；分布图展示最终 singlet AnnData 中的 QC。数值缺失保留为缺失，不补零。

In [ ]:
panels = [
    ("num_genes_expressed", "Sample_Unique_CpG_Sites", "RNA genes vs DNA CpG sites"),
    ("pct_counts_mt", "Sample_Meth_CpG_Rate%", "RNA mt% vs DNA CpG methylation%"),
    ("doublet_score", "Sample_Unique_CpG_Sites", "Doublet score vs DNA CpG sites"),
]
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for ax, (x, y, title) in zip(axes, panels):
    if qc.empty or x not in qc or y not in qc:
        ax.text(0.5, 0.5, "No paired QC data", ha="center", transform=ax.transAxes)
        ax.set_axis_off()
        continue
    plot = qc[[x, y, "predicted_doublet"]].copy()
    for col in (x, y):
        plot[col] = pd.to_numeric(plot[col], errors="coerce")
    sns.scatterplot(data=plot, x=x, y=y, hue="predicted_doublet", palette={False:"#4C78A8", True:"#E15759"},
                    s=12, alpha=0.55, linewidth=0, ax=ax)
    ax.set_title(title)
sns.despine(fig=fig)
save_plot(fig, "02_joint_qc_doublets")

metrics = ["num_genes_expressed", "pct_counts_mt", "Sample_Unique_CpG_Sites", "Sample_Meth_CpG_Rate%"]
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, col in zip(axes, metrics):
    if adata.n_obs and col in adata.obs:
        values = pd.to_numeric(adata.obs[col], errors="coerce").dropna()
        sns.histplot(values, bins=40, ax=ax, color="#59A14F")
        ax.set_title(f"Singlets: {col}")
    else:
        ax.set_axis_off()
sns.despine(fig=fig)
save_plot(fig, "03_singlet_qc")

## 方法依据与边界

- [SeekGene 官方双细胞教程](https://github.com/seekgene/SeekSoulMethyl_Downstream_Analysis/blob/23a8aa7db90a7798d6f5b0c0c7d49423568a8570/notebooks/en/scMethyl%2BRNA_Multi-omics_Doublet_Detection.ipynb) 区分 RNA 注释证据、DNA 深度提示和 MethylScrublet；本流程只输出 RNA Scrublet 预测。
- [Scanpy Scrublet](https://scanpy.readthedocs.io/en/stable/api/generated/scanpy.pp.scrublet.html) 在独立文库上模拟 doublet，不需要先运行细胞类型注释。
- HPC 参考 `…/SeekGene_merged_F12F/RNA/jikai_patient_adata.ipynb` 使用 Scrublet（先验 0.15）并结合人工簇注释；对应 DNA Notebook 人工删除指定 Leiden 簇。本流程不迁移人工簇号，也不把参考数据的 15% 当作所有项目的预期值。
- Lemmus 的输入为未取整的原生分子计数；保留原值用于模型，预测属于算法提示，不把预测当作人工双细胞真值。没有通过自动阈值判定的文库会报错，不强制输出 singlet。

### DNA doublet：甲基化计数分析与标记

使用当前 DNA RawAdata 的常染色体 1Mb 区域 mc/cov，逐文库运行 ALLCools MethylScrublet。DNA 阈值为自动阈值 × `DNA_DOUBLET_THRESHOLD_SCALE`（默认0.5）。这一步保留全部 DNA，RNA singlet 筛选仍只使用 RNA 判定。低信息 DNA 保持未判定；联合判定使用可空 OR：任一阳性为 True，仅双方阴性为 False。


In [ ]:
DNA_DOUBLET_DIR = processor.run_dna_doublet_analysis(
    RESULT_DIR, DNA_PROJECT_ROOT, doublet_threshold_scale=DNA_DOUBLET_THRESHOLD_SCALE)
dna_doublet_summary = json.loads((DNA_DOUBLET_DIR / "DNA_doublet_summary.json").read_text())
dna_doublets = pd.read_csv(DNA_DOUBLET_DIR / "DNA_doublet_scores.csv", dtype={"Sample_ID":str,"CloneID":str})
print(f"DNA cells: {dna_doublet_summary['dna_cells']:,}")
print(f"DNA doublets: {dna_doublet_summary['dna_doublets']:,}")
print(f"DNA 未判定: {dna_doublet_summary['dna_unassessed']:,}")
print(f"DNA 或 RNA 阳性: {dna_doublet_summary['joint_doublets']:,}")
print(DNA_DOUBLET_DIR)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for library, info in dna_doublet_summary["libraries"].items():
    values = dna_doublets.loc[dna_doublets.CloneID.eq(library), "DNA_doublet_score"].dropna()
    ax.hist(values, bins=60, alpha=.5, label=library)
    ax.axvline(info["auto_threshold"], color="grey", linestyle=":")
    ax.axvline(info["threshold"], color="red", linestyle="--")
ax.set(xlabel="DNA MethylScrublet score", ylabel="DNA cells")
ax.legend()
plt.show()


## 导出包含全部 QC 的 SingleCpG AnnData
保留原 SingleCpG 矩阵的全部 DNA 细胞（包括 doublet 和 DNA-only），保留原始矩阵、CpG 位点及已有注释。加入 DNA QC、带 `RNA_` 前缀的 RNA QC、doublet 判定和细胞对应信息；没有 RNA 的细胞保留缺失值。
输出到本次 `DNA_DOUBLET_DIR/SingleCpG_Adata.h5ad`。该步骤复制现有大文件，需要相应磁盘空间；不重新计算 CpG。

In [ ]:
SINGLECPG_FILE = processor.export_droplet_single_cpg(
    RESULT_DIR, DNA_PROJECT_ROOT, dna_doublet_result_dir=DNA_DOUBLET_DIR)
print(SINGLECPG_FILE)